# 🏦 リース審査 TimesFM バックエンド (Google Colab)

このノートブックは **ローカルの Streamlit アプリ** から呼び出される FastAPI サーバーを
Colab 上で起動し、ngrok で公開します。

### 使い方
1. ランタイム → **「T4 GPU」または「CPU」** を選択
2. 「すべてのセルを実行」
3. 最後のセルに表示される **ngrok URL** をコピー
4. ローカルの Streamlit アプリ「📊 3期財務分析」ページの
   **「⚙️ バックエンド設定」** にその URL を貼り付ける

In [ ]:
# ── セル 1: 依存パッケージのインストール ─────────────────────────────────
# timesfm: TimesFM 本体（torch バックエンド）
# fastapi + uvicorn: REST API サーバー
# pyngrok: ngrok トンネル（Colab からローカルへ公開）
!pip install -q timesfm fastapi uvicorn pyngrok nest-asyncio

In [ ]:
# ── セル 2: backend.py のコード（ローカルと同一） ─────────────────────────
# ローカルの backend.py をそのまま埋め込む
BACKEND_CODE = '''
from __future__ import annotations
import math
from datetime import date
import numpy as np
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, field_validator

# ── TimesFM 初期化 ────────────────────────────────────────────────────────
try:
    import timesfm
    import pandas as pd
    _TFM = timesfm.TimesFm(
        hparams=timesfm.TimesFmHparams(
            backend="gpu",        # Colab GPU を使用
            per_core_batch_size=32,
            horizon_len=128,
        ),
        checkpoint=timesfm.TimesFmCheckpoint(
            huggingface_repo_id="google/timesfm-1.0-200m",
        ),
    )
    TIMESFM_AVAILABLE = True
    print("✅ TimesFM 初期化完了")
except Exception as e:
    _TFM = None
    TIMESFM_AVAILABLE = False
    print(f"⚠️ TimesFM フォールバック: {e}")

# ── 業種別季節性インデックス ────────────────────────────────────────────
SEASONAL_INDICES = {
    "建設業":       [0.6, 0.7, 1.8, 0.8, 0.9, 0.9, 0.8, 0.9, 1.0, 0.9, 1.0, 1.7],
    "小売業":       [0.9, 0.8, 1.0, 0.9, 0.9, 0.9, 1.0, 1.0, 0.9, 1.0, 1.1, 1.6],
    "製造業":       [0.9, 0.9, 1.1, 1.0, 1.0, 1.0, 1.0, 0.9, 1.1, 1.0, 1.0, 1.1],
    "卸売業":       [0.9, 0.9, 1.2, 1.0, 1.0, 1.0, 1.0, 0.9, 1.0, 1.0, 1.0, 1.1],
    "医療・福祉":   [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    "飲食・宿泊業": [0.8, 0.8, 1.0, 1.0, 1.1, 1.1, 1.3, 1.2, 1.0, 1.0, 0.9, 0.8],
    "サービス業":   [0.9, 0.9, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.1],
    "不動産業":     [0.8, 0.9, 1.4, 1.1, 1.0, 0.9, 0.9, 0.9, 1.0, 0.9, 0.9, 1.3],
    "情報通信業":   [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    "運輸・物流":   [0.9, 0.9, 1.1, 1.0, 1.0, 0.9, 0.9, 1.0, 1.0, 1.0, 1.1, 1.2],
}
_DEFAULT_SEASONAL = [1.0] * 12

app = FastAPI(title="リース審査 TimesFM API (Colab)")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class ForecastRequest(BaseModel):
    sales:      list[float]
    profit:     list[float]
    net_assets: list[float]
    industry:   str = "サービス業"

    @field_validator("sales", "profit", "net_assets")
    @classmethod
    def must_be_three(cls, v):
        if len(v) != 3:
            raise ValueError("3要素必要")
        return v

class ForecastResponse(BaseModel):
    months_history: list[str]
    sales_history: list[float]
    profit_history: list[float]
    net_assets_history: list[float]
    months_forecast: list[str]
    sales_forecast: list[float]
    profit_forecast: list[float]
    net_assets_forecast: list[float]
    timesfm_available: bool

def _annual_to_monthly(annual, industry):
    idx = SEASONAL_INDICES.get(industry, _DEFAULT_SEASONAL)
    return [annual[i // 12] / 12.0 * idx[i % 12] for i in range(36)]

def _timesfm_forecast(history, horizon=12):
    if not TIMESFM_AVAILABLE or _TFM is None:
        return _gbm_forecast(history, horizon)
    try:
        df = pd.DataFrame({"unique_id": ["x"]*len(history), "ds": range(len(history)), "y": history})
        fdf, _ = _TFM.forecast_on_df(inputs=df, freq="M", value_name="y", num_jobs=1)
        col = [c for c in fdf.columns if "timesfm" in c.lower()]
        if col:
            return fdf[col[0]].values[:horizon].tolist()
    except Exception as e:
        print(f"TimesFM forecast error: {e}")
    return _gbm_forecast(history, horizon)

def _gbm_forecast(history, horizon=12):
    if not history or history[-1] <= 0:
        return [0.0] * horizon
    arr = np.clip(history, 1e-6, None)
    log_r = np.diff(np.log(arr))
    mu = float(np.mean(log_r)) if len(log_r) > 0 else 0.01
    S0 = history[-1]
    return [S0 * math.exp(mu * (t+1) / 12) for t in range(horizon)]

def _month_labels(y, m, n):
    labels = []
    for _ in range(n):
        labels.append(f"{y:04d}-{m:02d}")
        m += 1
        if m > 12: m, y = 1, y+1
    return labels

@app.get("/health")
def health():
    return {"status": "ok", "timesfm": TIMESFM_AVAILABLE, "backend": "colab"}

@app.post("/forecast", response_model=ForecastResponse)
def forecast(req: ForecastRequest):
    today = date.today()
    hist_labels = _month_labels(today.year-3, today.month, 36)
    fore_labels  = _month_labels(today.year,   today.month, 12)
    sh = _annual_to_monthly(req.sales,      req.industry)
    ph = _annual_to_monthly(req.profit,     req.industry)
    nh = _annual_to_monthly(req.net_assets, req.industry)
    return ForecastResponse(
        months_history=hist_labels, sales_history=sh, profit_history=ph, net_assets_history=nh,
        months_forecast=fore_labels,
        sales_forecast=_timesfm_forecast(sh),
        profit_forecast=_timesfm_forecast(ph),
        net_assets_forecast=_timesfm_forecast(nh),
        timesfm_available=TIMESFM_AVAILABLE,
    )
'''

# ファイルに書き出す
with open('/content/backend_colab.py', 'w') as f:
    f.write(BACKEND_CODE)
print('backend_colab.py を作成しました')

In [ ]:
# ── セル 3: ngrok トークンの設定 ──────────────────────────────────────────
# https://dashboard.ngrok.com/get-started/your-authtoken から取得
# 無料アカウントで OK（月 1GB まで）
NGROK_TOKEN = ""  # ← ここに ngrok トークンを貼り付ける

from pyngrok import ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    print('✅ ngrok トークン設定済み')
else:
    print('⚠️ NGROK_TOKEN が未設定です。トークンなしでも動作しますが接続が不安定になることがあります。')

In [ ]:
# ── セル 4: FastAPI サーバーを起動して ngrok で公開 ───────────────────────
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok

# Colab の Jupyter ループと uvicorn を共存させるために必要
nest_asyncio.apply()

# backend_colab.py を読み込む
import importlib.util, sys
spec = importlib.util.spec_from_file_location('backend_colab', '/content/backend_colab.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
app = mod.app

# uvicorn をバックグラウンドスレッドで起動
PORT = 8000
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='warning')

t = threading.Thread(target=run_server, daemon=True)
t.start()

# ngrok でトンネルを開く
import time; time.sleep(2)  # サーバー起動待ち
tunnel = ngrok.connect(PORT, bind_tls=True)
public_url = tunnel.public_url

print('=' * 60)
print('✅ TimesFM バックエンド起動完了！')
print()
print(f'  公開URL: {public_url}')
print()
print('📋 Streamlit アプリへの設定手順:')
print('  1. ローカルの Streamlit で「📊 3期財務分析」を開く')
print('  2.「⚙️ バックエンド設定」を展開する')
print(f'  3. 以下の URL を貼り付ける:')
print(f'     {public_url}')
print('=' * 60)

In [ ]:
# ── セル 5: 接続テスト（オプション） ─────────────────────────────────────
import requests

# /health チェック
r = requests.get(f'{public_url}/health')
print('/health:', r.json())

# /forecast テスト
r2 = requests.post(f'{public_url}/forecast', json={
    'sales':      [500000, 520000, 550000],
    'profit':     [30000,  35000,  38000],
    'net_assets': [120000, 145000, 170000],
    'industry':   '建設業'
})
d = r2.json()
print(f'/forecast: status={r2.status_code}, timesfm={d["timesfm_available"]}')
print(f'  sales_forecast[-1]: {d["sales_forecast"][-1]:,.0f} 千円')

In [ ]:
# ── セル 6: サーバーを止めるとき ─────────────────────────────────────────
# このセルを実行するとトンネルが閉じる
# ngrok.kill()
# print('サーバーを停止しました')